# MvDEC 2025 Equation 11 literal reproduction: REUTERS seed 42

Runs the immutable `mvdec_2025_equation11_literal_v1` protocol on Kaggle GPU, validates its TensorFlow contract, audits the saved artifact, compares View1/View2/fused metrics with the MvDEC paper, and creates a downloadable ZIP.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a GPU accelerator in Kaggle notebook settings."


In [ ]:
repo_url = "https://github.com/vdat-18/mvdec_fpmax.git"
repo_dir = "/kaggle/working/mvdec_fpmax"
!rm -rf {repo_dir}
!git clone --branch dev --single-branch {repo_url} {repo_dir}
%cd {repo_dir}
!python -m pip install -q -e . gdown


In [ ]:
!python scripts/fetch_dekm_release.py --datasets REUTERS
!pytest -q tests/test_mvdec_preprocessing.py \
    -k "literal_equation11 or public_reproduction_artifact"


In [ ]:
output_dir = "/kaggle/working/mvdec-equation11"
!PYTHONHASHSEED=42 python -u src/representation_learning/MVDEC_dense.py REUTERS \
    --runs 1 \
    --seed 42 \
    --protocol mvdec_2025_equation11_literal_v1 \
    --lambda-kmeans 1 \
    --progress-interval 100 \
    --dataset-root external_repos/DEKM/datasets \
    --public-output-dir {output_dir}


In [ ]:
!mvdec-audit-runs --root {output_dir}


In [ ]:
import json
from pathlib import Path

import pandas as pd

protocol_id = "mvdec_2025_equation11_literal_v1"
protocol_root = Path(output_dir) / "reuters" / protocol_id
config_root = next(path for path in protocol_root.iterdir() if path.is_dir())
run_dir = config_root / "seed_42"
runs = pd.read_csv(config_root / "mvdec_runs.csv")
summary = pd.read_csv(config_root / "mvdec_summary.csv")
manifest = json.loads((run_dir / "manifest.json").read_text())

metric_columns = [
    "random_seed", "view1_acc", "view1_nmi", "view2_acc", "view2_nmi",
    "fused_acc", "fused_nmi", "runtime_seconds",
]
display(runs[metric_columns])
display(summary)


In [ ]:
paper = {
    "view1_acc": 0.7642,
    "view1_nmi": 0.5999,
    "view2_acc": 0.6501,
    "view2_nmi": 0.5129,
    "fused_acc": 0.7813,
    "fused_nmi": 0.6141,
}
row = runs.iloc[0]
comparison = pd.DataFrame([
    {
        "metric": metric,
        "reproduction_percent": float(row[metric]) * 100,
        "paper_percent": reference * 100,
        "difference_pp": (float(row[metric]) - reference) * 100,
    }
    for metric, reference in paper.items()
])
display(comparison)


In [ ]:
contract = manifest["config"]["protocol_contract"]
print("Protocol:", manifest["protocol_id"])
print("Config hash:", manifest["config_hash"])
print("Dataset SHA-256:", manifest["config"]["source_sha256"])
print("Runtime seconds:", manifest["runtime_seconds"])
print("Status:", manifest["status"])
print("Loss contract:")
print(json.dumps(contract["objective"], indent=2))
print("Decision provenance:")
print(json.dumps(contract["decision_provenance"], indent=2))
print("Training log tail:")
print("\n".join((run_dir / "training_log.csv").read_text().splitlines()[-12:]))


In [ ]:
import shutil

zip_path = "/kaggle/working/mvdec-equation11-reuters-seed42"
shutil.make_archive(zip_path, "zip", "/kaggle/working", "mvdec-equation11")
print("Result:", zip_path + ".zip")
